# Step 3: RAGAS Judge Validation for Penilaian Makalah

## Tahapan Pipeline:
1. **Setup & Initialization** - Configure RAG, LLM, environment
2. **Stage 1** - Retrieve Assessment Context (via LightRAG)
3. **Stage 2** - Evaluate Paper (Evaluator LLM)
4. **Stage 3** - RAGAS Metrics (Faithfulness & Answer Relevance)
5. **Stage 4** - Judge Validation & Final Scoring
6. **Full Pipeline** - End-to-end orchestration
7. **Results** - Visualization & Export (JSON, CSV)

---

## Cell 1: Imports & Setup

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
import sys
import asyncio
import json
import logging
import tempfile
import re
from pathlib import Path
from xml.etree import ElementTree as ET
from datetime import datetime
import zipfile

import pandas as pd
import numpy as np
from dotenv import load_dotenv

# LightRAG
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete_if_cache, openai_embed
from lightrag.utils import wrap_embedding_func_with_attrs, setup_logger, set_verbose_debug

# PDF reading
try:
    import pdfplumber
except ImportError:
    print("⚠️ pdfplumber tidak terinstall. Install dengan: pip install pdfplumber")

# Configure logging
logging.basicConfig(level=logging.INFO)
log = logging.getLogger("penilaian_makalah_ragas")

print("✅ Imports berhasil dimuat")

## Cell 2: Environment & Constants

In [ ]:
# Load .env
env_path = os.path.abspath(os.path.join(os.path.dirname("."), "../.env"))
if os.path.exists(env_path):
    load_dotenv(dotenv_path=env_path, override=True)
else:
    load_dotenv(override=True)

# Constants
WORKING_DIR = os.getenv("LIGHTRAG_WORKING_DIR", "./rag_storage")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4-mini")
LLM_API_KEY = os.getenv("LLM_BINDING_API_KEY", "")
LLM_BASE_URL = os.getenv("LLM_BINDING_HOST", "http://localhost:8000/v1")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIM = int(os.getenv("EMBEDDING_DIM", 1536))
EMBEDDING_TOKEN_LIMIT = int(os.getenv("EMBEDDING_TOKEN_LIMIT", 8192))

# Scoring constants
SCORE_LABELS = {
    "n1_kesesuaian_judul": "Kesesuaian Judul dengan Tema",
    "n2_kesesuaian_isi": "Kesesuaian Isi dengan Judul & Tema",
    "n3_sistematika": "Sistematika Penulisan",
    "n4_ketajaman_analisis": "Ketajaman Analisis",
    "n5_penggunaan_bahasa": "Penggunaan Bahasa",
}
SCORE_WEIGHTS = {
    "n1_kesesuaian_judul": 1,
    "n2_kesesuaian_isi": 1,
    "n3_sistematika": 1,
    "n4_ketajaman_analisis": 2,
    "n5_penggunaan_bahasa": 1,
}

print(f"✅ Environment loaded")
print(f"   LLM: {LLM_MODEL}")
print(f"   Host: {LLM_BASE_URL}")
print(f"   Embedding: {EMBEDDING_MODEL} (dim={EMBEDDING_DIM})")

## Cell 3: Document Parsers (DOCX, PDF, TXT)

In [ ]:
NS = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
VMERGE = "{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val"


def _read_docx(filepath: str) -> str:
    """Extract text from DOCX file."""
    with zipfile.ZipFile(filepath) as z:
        root = ET.fromstring(z.read("word/document.xml"))
    lines = []
    for child in root.find(".//w:body", NS):
        tag = child.tag.split("}")[-1]
        if tag == "p":
            text = "".join(
                r.find("w:t", NS).text for r in child.findall(".//w:r", NS)
                if r.find("w:t", NS) is not None and r.find("w:t", NS).text
            ).strip()
            if text:
                lines.append(text)
        elif tag == "tbl":
            for row in child.findall("w:tr", NS):
                cells = []
                for cell in row.findall("w:tc", NS):
                    vm = cell.find(".//w:vMerge", NS)
                    if vm is not None and vm.get(VMERGE) != "restart":
                        continue
                    text = "".join(
                        r.find("w:t", NS).text
                        for p in cell.findall(".//w:p", NS)
                        for r in p.findall(".//w:r", NS)
                        if r.find("w:t", NS) is not None and r.find("w:t", NS).text
                    ).strip()
                    if text:
                        cells.append(text)
                if cells:
                    lines.append(" | ".join(cells))
    return "\n".join(lines)


def _read_pdf(filepath: str) -> str:
    """Extract text from PDF file."""
    try:
        with pdfplumber.open(filepath) as pdf:
            return "\n".join(p.extract_text() for p in pdf.pages if p.extract_text())
    except ImportError:
        return "[pdfplumber tidak terinstall]"


def extract_text_from_file(filepath: str) -> str:
    """Extract text dari file (DOCX, PDF, TXT)."""
    suffix = Path(filepath).suffix.lower()
    if suffix == ".docx":
        return _read_docx(filepath)
    elif suffix == ".pdf":
        return _read_pdf(filepath)
    elif suffix == ".txt":
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()
    return ""


def read_folder_documents(folder_path: str) -> list:
    """Read all documents dari folder."""
    results = []
    for path in Path(folder_path).rglob("*"):
        suffix = path.suffix.lower()
        if suffix in [".docx", ".pdf", ".txt"]:
            text = extract_text_from_file(str(path))
            if text.strip():
                results.append({"filename": path.name, "text": text})
                print(f"  ✓ {path.name} ({len(text):,} karakter)")
    return results


print("✅ Document parsers ready")

## Cell 4: Initialize LightRAG

In [ ]:
async def llm_model_func(
    prompt, system_prompt=None, history_messages=[], keyword_extraction=False, **kwargs
) -> str:
    """LLM function untuk LightRAG."""
    return await openai_complete_if_cache(
        LLM_MODEL,
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=LLM_API_KEY,
        base_url=LLM_BASE_URL,
        **kwargs,
    )


async def initialize_rag() -> LightRAG:
    """Initialize LightRAG instance."""
    async def raw_embedding_func(texts):
        return await openai_embed.func(
            texts,
            api_key=LLM_API_KEY,
            base_url=LLM_BASE_URL,
            model=EMBEDDING_MODEL,
        )

    embedding_func = wrap_embedding_func_with_attrs(
        embedding_dim=EMBEDDING_DIM,
        max_token_size=EMBEDDING_TOKEN_LIMIT,
        model_name=EMBEDDING_MODEL,
    )(raw_embedding_func)

    rag = LightRAG(
        working_dir=WORKING_DIR,
        llm_model_func=llm_model_func,
        embedding_func=embedding_func,
        kv_storage="LocalKVStorage",
        vector_storage="LocalVectorStorage",
        graph_storage="LocalGraphStorage",
    )
    
    await rag.initialize_storages()
    return rag


def run_async(coro):
    """Run async coroutine synchronously."""
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        return loop.run_until_complete(coro)
    finally:
        loop.close()


print("✅ LightRAG setup ready")

## Cell 5: Prompts & JSON Parsing

In [ ]:
QUERY_KEYWORDS = """
Penilaian Penulisan Makalah Form. 1 Penilaian Penulisan Makalah {selected_jabatan}
Kesesuaian Judul dengan Tema Kesesuaian Isi Makalah dengan Judul dan Tema
Sistematika Penulisan Ketajaman Analisis Penggunaan Bahasa dalam Penulisan Makalah
Bobot Penilaian Format Penulisan Struktur Makalah Penilaian Kompetensi Teknis
"""

PROMPT_KONTEKS = """
Anda adalah asisten yang bertugas mengumpulkan konteks relevan untuk penilaian makalah.
Berdasarkan jabatan '{selected_jabatan}', berikan ringkasan singkat tentang:
1. Deskripsi jabatan dan kompetensi yang diperlukan
2. Kriteria penilaian utama untuk posisi ini
3. Standar kualitas yang diharapkan dalam penulisan makalah
4. Rencana Strategis BPOM yang relevan untuk menilai kedalaman analisis
Berikan jawaban dalam format paragraf singkat, fokus pada poin-poin penting.
"""

PROMPT_PENILAIAN = """
---Role---
Anda adalah evaluator akademik sebagai Panitia Seleksi yang bertugas menilai kualitas substansi makalah secara objektif dan sistematis.

---Goal---
Melakukan penilaian terhadap makalah berdasarkan kriteria penilaian yang telah ditentukan, memberikan skor numerik untuk setiap kriteria, serta menyusun justifikasi yang jelas dan berbasis bukti dari isi makalah.

---Konteks Jabatan---
{assessment_context}

---Ketentuan Penulisan Makalah (Tema)---
{tema_text}

---Instructions---
1. Baca dan pahami isi makalah secara menyeluruh.
2. Tinjau konteks jabatan di atas sebagai acuan penilaian.
3. Lakukan penilaian terhadap setiap kriteria dengan memberikan skor antara 40 sampai 100.
4. Setiap skor harus disertai justifikasi yang menjelaskan alasan pemberian skor.
5. Penilaian harus objektif, sistematis, dan berbasis isi makalah.
6. Gunakan bahasa formal dan akademik.
7. Jangan menggunakan informasi di luar isi makalah.
8. Jika informasi dalam makalah terbatas, tetap berikan skor dengan menjelaskan keterbatasan informasi tersebut.
9. Hitung nilai akhir menggunakan rumus yang telah ditentukan.
10. Output harus dalam format JSON yang valid dan tidak boleh mengandung teks tambahan di luar JSON.

---Assessment Criteria---
1. Kesesuaian judul dengan tema
2. Kesesuaian isi makalah dengan judul dan tema
3. Sistematika penulisan
4. Ketajaman analisis (bobot 2x)
5. Penggunaan bahasa dalam penulisan makalah

---Scoring Rules---
- Skor minimum: 40, maksimum: 100, harus bilangan bulat
- Ketajaman analisis memiliki bobot dua kali lipat dalam nilai akhir

---Makalah---
{makalah_text}

---Output Format---
Output MUST be a valid JSON format. All property names and string values MUST be enclosed in double quotes. Do not use trailing commas. Do not wrap the JSON in markdown blocks.
Example output format:
{
  "Ringkasan": "ringkasan isi makalah secara keseluruhan",
  "scores": {
    "n1_kesesuaian_judul": 0,
    "n2_kesesuaian_isi": 0,
    "n3_sistematika": 0,
    "n4_ketajaman_analisis": 0,
    "n5_penggunaan_bahasa": 0
  },
  "justification": {
    "n1_kesesuaian_judul": "",
    "n2_kesesuaian_isi": "",
    "n3_sistematika": "",
    "n4_ketajaman_analisis": "",
    "n5_penggunaan_bahasa": ""
  },
  "evidence": {
    "n1_kesesuaian_judul": "",
    "n2_kesesuaian_isi": "",
    "n3_sistematika": "",
    "n4_ketajaman_analisis": "",
    "n5_penggunaan_bahasa": ""
  },
  "final_score": 0
}
"""


def parse_json_response(raw: str) -> dict:
    """Extract JSON dari LLM response."""
    raw = raw.strip()
    # Strip markdown if present
    if raw.startswith("```"):
        parts = raw.split("```")
        raw = parts[1] if len(parts) > 1 else raw
        if raw.startswith("json"):
            raw = raw[4:]
    
    raw = raw.strip()
    # Extract JSON block
    start = raw.find('{')
    end = raw.rfind('}')
    if start != -1 and end != -1:
        raw = raw[start:end+1]
    
    # Remove trailing commas
    raw = re.sub(r',\s*([\]}])', r'\1', raw)
    
    return json.loads(raw)


def compute_final_score(scores: dict) -> float:
    """Compute weighted final score."""
    total_weight = sum(SCORE_WEIGHTS.values())
    weighted_sum = sum(scores.get(k, 0) * w for k, w in SCORE_WEIGHTS.items())
    return round(weighted_sum / total_weight, 1)


print("✅ Prompts & parsing ready")

## Cell 6: RAGAS Metrics - Faithfulness Evaluator

In [ ]:
PROMPT_FAITHFULNESS = """
---Role---
Anda adalah expert evaluator yang mengevaluasi faithfulness (kesetiaan) dari evaluasi makalah terhadap isi makalah yang sebenarnya.

---Task---
Evaluasi apakah skor dan justifikasi yang diberikan oleh evaluator DIDUKUNG SEPENUHNYA oleh bukti dalam makalah. Berikan confidence score 0-1.

---Context---
**Makalah:**
{makalah}

**Kriteria:** {kriteria}

**Evaluasi (Score dan Justifikasi):**
Skor: {skor}
Justifikasi: {justifikasi}
Bukti yang dikutip: {evidence}

---Evaluation Questions---
1. Apakah bukti yang dikutip benar-benar ada dan sesuai dalam makalah?
2. Apakah skor {skor} logis berdasarkan justifikasi yang diberikan?
3. Apakah justifikasi tidak menambahkan informasi atau interpretasi di luar makalah?
4. Apakah ada bukti yang bertentangan dalam makalah?

---Output Format---
Output MUST be valid JSON:
{{
  "faithful": true/false,
  "confidence": 0.0-1.0,
  "issues": ["issue1", "issue2"],
  "explanation": "penjelasan singkat"
}}
"""


async def evaluate_faithfulness(
    rag: LightRAG,
    makalah: str,
    kriteria: str,
    skor: int,
    justifikasi: str,
    evidence: str,
) -> dict:
    """Evaluate faithfulness of evaluation against paper."""
    prompt = PROMPT_FAITHFULNESS.format(
        makalah=makalah[:2000],
        kriteria=kriteria,
        skor=skor,
        justifikasi=justifikasi,
        evidence=evidence,
    )
    
    response = await rag.llm_model_func(prompt)
    try:
        result = parse_json_response(response)
        return {
            "faithful": result.get("faithful", False),
            "confidence": result.get("confidence", 0.5),
            "issues": result.get("issues", []),
            "explanation": result.get("explanation", ""),
        }
    except Exception as e:
        log.warning(f"Faithfulness parsing error: {e}")
        return {
            "faithful": False,
            "confidence": 0.3,
            "issues": ["Parsing error"],
            "explanation": str(e),
        }


print("✅ Faithfulness evaluator ready")

## Cell 7: RAGAS Metrics - Answer Relevance Evaluator

In [ ]:
PROMPT_ANSWER_RELEVANCE = """
---Role---
Anda adalah expert evaluator yang mengevaluasi answer relevance (relevansi jawaban) dari evaluasi terhadap kriteria penilaian.

---Task---
Evaluasi apakah skor dan justifikasi yang diberikan RELEVAN dan MENJAWAB kriteria penilaian dengan tepat. Berikan confidence score 0-1.

---Context---
**Kriteria Penilaian:** {kriteria}
**Deskripsi Kriteria:** {deskripsi_kriteria}

**Evaluasi:**
Skor: {skor}
Justifikasi: {justifikasi}

**Rincian Makalah (excerpt):** {paper_excerpt}

---Evaluation Questions---
1. Apakah justifikasi secara langsung mengevaluasi kriteria tersebut?
2. Apakah justifikasi menjelaskan MENGAPA skor tersebut diberikan?
3. Apakah skor sesuai dengan deskripsi kriteria?
4. Apakah evaluasi komprehensif mencakup aspek-aspek penting dari kriteria?

---Output Format---
Output MUST be valid JSON:
{{
  "relevant": true/false,
  "confidence": 0.0-1.0,
  "gaps": ["gap1", "gap2"],
  "suggestions": ["saran1", "saran2"],
  "explanation": "penjelasan singkat"
}}
"""

CRITERIA_DESCRIPTIONS = {
    "n1_kesesuaian_judul": "Judul harus sesuai dengan tema penulisan makalah yang ditentukan",
    "n2_kesesuaian_isi": "Isi makalah harus sesuai dengan judul dan tema yang dikemukakan",
    "n3_sistematika": "Makalah harus memiliki struktur logis: pendahuluan, analisis, rencana strategis, rencana aksi, kesimpulan",
    "n4_ketajaman_analisis": "Analisis harus mendalam, kritis, dan menunjukkan pemahaman terhadap isu strategis",
    "n5_penggunaan_bahasa": "Bahasa harus formal, akademik, dan mudah dipahami tanpa kesalahan ketik",
}


async def evaluate_answer_relevance(
    rag: LightRAG,
    kriteria_key: str,
    justifikasi: str,
    paper_excerpt: str,
    skor: int,
) -> dict:
    """Evaluate relevance of evaluation to criteria."""
    prompt = PROMPT_ANSWER_RELEVANCE.format(
        kriteria=SCORE_LABELS.get(kriteria_key, kriteria_key),
        deskripsi_kriteria=CRITERIA_DESCRIPTIONS.get(kriteria_key, ""),
        skor=skor,
        justifikasi=justifikasi,
        paper_excerpt=paper_excerpt[:1000],
    )
    
    response = await rag.llm_model_func(prompt)
    try:
        result = parse_json_response(response)
        return {
            "relevant": result.get("relevant", False),
            "confidence": result.get("confidence", 0.5),
            "gaps": result.get("gaps", []),
            "suggestions": result.get("suggestions", []),
            "explanation": result.get("explanation", ""),
        }
    except Exception as e:
        log.warning(f"Answer Relevance parsing error: {e}")
        return {
            "relevant": False,
            "confidence": 0.3,
            "gaps": ["Parsing error"],
            "suggestions": [],
            "explanation": str(e),
        }


print("✅ Answer Relevance evaluator ready")

## Cell 8: Judge Validator with RAGAS

In [ ]:
async def validate_evaluation_with_ragas(
    rag: LightRAG,
    makalah: str,
    evaluation_result: dict,
) -> dict:
    """
    Validate evaluation result using RAGAS metrics.
    Returns confidence scores and flags untuk criteria yang perlu review.
    """
    scores = evaluation_result.get("scores", {})
    justification = evaluation_result.get("justification", {})
    evidence = evaluation_result.get("evidence", {})
    
    validation_result = {
        "timestamp": datetime.now().isoformat(),
        "confidence_scores": {},
        "ragas_metrics": {},
        "flagged_criteria": [],
        "overall_quality_score": 0.0,
        "judge_notes": "",
    }
    
    # Evaluate each criterion
    all_confidences = []
    for criterion_key in SCORE_LABELS.keys():
        criterion_label = SCORE_LABELS[criterion_key]
        score = scores.get(criterion_key, 0)
        just = justification.get(criterion_key, "")
        evid = evidence.get(criterion_key, "")
        
        # Get paper excerpt for context
        paper_excerpt = makalah[:1500] if len(makalah) > 1500 else makalah
        
        # Evaluate Faithfulness
        print(f"  📊 Evaluating Faithfulness for {criterion_key}...")
        faith_result = await evaluate_faithfulness(
            rag,
            makalah,
            criterion_label,
            score,
            just,
            evid,
        )
        
        # Evaluate Answer Relevance
        print(f"  📊 Evaluating Answer Relevance for {criterion_key}...")
        relevance_result = await evaluate_answer_relevance(
            rag,
            criterion_key,
            just,
            paper_excerpt,
            score,
        )
        
        # Compute combined confidence
        faith_conf = faith_result.get("confidence", 0.5)
        relevance_conf = relevance_result.get("confidence", 0.5)
        combined_confidence = (faith_conf + relevance_conf) / 2
        
        validation_result["confidence_scores"][criterion_key] = round(combined_confidence, 2)
        validation_result["ragas_metrics"][criterion_key] = {
            "faithfulness": round(faith_conf, 2),
            "answer_relevance": round(relevance_conf, 2),
            "faith_issues": faith_result.get("issues", []),
            "relevance_gaps": relevance_result.get("gaps", []),
        }
        
        all_confidences.append(combined_confidence)
        
        # Flag if confidence too low
        if combined_confidence < 0.6:
            validation_result["flagged_criteria"].append({
                "criterion": criterion_key,
                "reason": f"Low confidence ({combined_confidence:.2f})",
                "faith_score": round(faith_conf, 2),
                "relevance_score": round(relevance_conf, 2),
                "issues": faith_result.get("issues", []) + relevance_result.get("gaps", []),
            })
    
    # Overall quality score
    if all_confidences:
        validation_result["overall_quality_score"] = round(np.mean(all_confidences), 2)
    
    # Judge notes
    num_flagged = len(validation_result["flagged_criteria"])
    if num_flagged == 0:
        validation_result["judge_notes"] = "✅ Evaluasi VALID - Semua kriteria memiliki confidence tinggi"
    elif num_flagged <= 2:
        validation_result["judge_notes"] = f"⚠️ PERHATIAN - {num_flagged} kriteria memiliki confidence rendah, perlu ditinjau"
    else:
        validation_result["judge_notes"] = f"🔴 CAUTION - {num_flagged} kriteria perlu review, kualitas evaluasi rendah"
    
    return validation_result


print("✅ Judge Validator ready")

## Cell 9: Stage 1 - Retrieve Assessment Context

In [ ]:
async def retrieve_assessment_context(
    rag: LightRAG,
    selected_jabatan: str,
    query_mode: str = "hybrid",
) -> str:
    """
    Stage 1: Retrieve konteks jabatan dari knowledge base.
    Returns assessment context untuk digunakan di evaluasi.
    """
    print(f"\n🔍 Stage 1: Retrieving assessment context for '{selected_jabatan}'...")
    
    try:
        context_response = await rag.aquery(
            query=QUERY_KEYWORDS.format(selected_jabatan=selected_jabatan),
            param=QueryParam(
                mode=query_mode,
                user_prompt=PROMPT_KONTEKS.format(selected_jabatan=selected_jabatan),
            ),
        )
        context = context_response if context_response else "Konteks jabatan tidak ditemukan dalam knowledge base."
        print(f"✅ Context retrieved ({len(context)} karakter)")
        return context
    except Exception as e:
        print(f"❌ Error: {e}")
        return f"Error retrieving context: {str(e)}"

print("✅ Stage 1 ready")

## Cell 10: Stage 2 - Evaluate Paper

In [ ]:
async def evaluate_paper_with_context(
    rag: LightRAG,
    makalah_text: str,
    assessment_context: str,
    tema_text: str = "",
    query_mode: str = "hybrid",
) -> dict:
    """
    Stage 2: Evaluate makalah berdasarkan assessment context.
    Returns structured evaluation dengan scores, justifications, evidence.
    """
    print(f"\n📝 Stage 2: Evaluating paper...")
    
    evaluation_prompt = PROMPT_PENILAIAN.format(
        assessment_context=assessment_context,
        makalah_text=makalah_text,
        tema_text=tema_text or "Tidak ada tema khusus",
    )
    
    try:
        eval_response = await rag.llm_model_func(evaluation_prompt)
        result = parse_json_response(eval_response)
        
        # Ensure final_score is computed
        if "final_score" not in result or result["final_score"] == 0:
            scores = result.get("scores", {})
            result["final_score"] = compute_final_score(scores)
        
        print(f"✅ Evaluation complete (Score: {result.get('final_score', 'N/A')})")
        return result
    except Exception as e:
        print(f"❌ Evaluation error: {e}")
        return {"error": str(e), "scores": {}}

print("✅ Stage 2 ready")

## Cell 11: Stage 3 - RAGAS Validation

In [ ]:
async def stage_3_ragas_validation(
    rag: LightRAG,
    makalah: str,
    evaluation_result: dict,
) -> dict:
    """
    Stage 3: Run RAGAS metrics on evaluation result.
    Evaluates Faithfulness & Answer Relevance of the evaluation.
    """
    print(f"\n🔬 Stage 3: Running RAGAS validation...")
    
    validation = await validate_evaluation_with_ragas(rag, makalah, evaluation_result)
    
    print(f"✅ RAGAS validation complete")
    print(f"   Overall Quality Score: {validation['overall_quality_score']}")
    print(f"   Flagged Criteria: {len(validation['flagged_criteria'])}")
    
    return validation

print("✅ Stage 3 ready")

## Cell 12: Stage 4 - Judge Validation & Final Scoring

In [ ]:
async def stage_4_judge_validation(
    evaluation_result: dict,
    validation_result: dict,
) -> dict:
    """
    Stage 4: Combine evaluation + RAGAS validation untuk final judge result.
    """
    print(f"\n⚖️ Stage 4: Judge Validation & Final Scoring...")
    
    final_result = {
        "timestamp": datetime.now().isoformat(),
        "evaluation": evaluation_result,
        "judge_validation": validation_result,
        "final_recommendation": "",
        "revision_needed": False,
    }
    
    # Determine final recommendation
    quality_score = validation_result.get("overall_quality_score", 0)
    num_flagged = len(validation_result.get("flagged_criteria", []))
    
    if quality_score >= 0.85 and num_flagged == 0:
        final_result["final_recommendation"] = "✅ APPROVED - Evaluasi valid dan reliable"
        final_result["revision_needed"] = False
    elif quality_score >= 0.70 and num_flagged <= 1:
        final_result["final_recommendation"] = "⚠️ APPROVED WITH NOTES - Evaluasi mostly valid"
        final_result["revision_needed"] = False
    elif quality_score >= 0.60:
        final_result["final_recommendation"] = "⚠️ NEEDS REVIEW - Evaluasi perlu tinjauan ulang"
        final_result["revision_needed"] = True
    else:
        final_result["final_recommendation"] = "🔴 NOT APPROVED - Evaluasi kualitas rendah"
        final_result["revision_needed"] = True
    
    print(f"   Recommendation: {final_result['final_recommendation']}")
    
    return final_result

print("✅ Stage 4 ready")

## Cell 13: Full Pipeline - End-to-End Workflow

In [ ]:
async def full_pipeline(
    rag: LightRAG,
    makalah_text: str,
    tema_text: str = "",
    selected_jabatan: str = "Sekretaris Utama",
    query_mode: str = "hybrid",
) -> dict:
    """
    Full pipeline: Context retrieval → Evaluation → RAGAS validation → Judge result.
    """
    print("\n" + "="*80)
    print("🚀 STARTING FULL PIPELINE: LLM-as-Judge + RAGAS Validation")
    print("="*80)
    
    start_time = datetime.now()
    
    # Stage 1: Retrieve Assessment Context
    assessment_context = await retrieve_assessment_context(rag, selected_jabatan, query_mode)
    
    # Stage 2: Evaluate Paper
    evaluation_result = await evaluate_paper_with_context(
        rag, makalah_text, assessment_context, tema_text, query_mode
    )
    
    if "error" in evaluation_result:
        print(f"❌ Pipeline failed at Stage 2: {evaluation_result['error']}")
        return evaluation_result
    
    # Stage 3: RAGAS Validation
    validation_result = await stage_3_ragas_validation(rag, makalah_text, evaluation_result)
    
    # Stage 4: Judge Validation & Final Scoring
    final_result = await stage_4_judge_validation(evaluation_result, validation_result)
    
    # Add timing
    elapsed = (datetime.now() - start_time).total_seconds()
    final_result["processing_time_seconds"] = elapsed
    
    print("\n" + "="*80)
    print("✅ PIPELINE COMPLETE")
    print(f"   Processing Time: {elapsed:.2f}s")
    print(f"   Final Score: {evaluation_result.get('final_score', 'N/A')}")
    print(f"   Quality Score: {validation_result.get('overall_quality_score', 'N/A')}")
    print(f"   Status: {final_result['final_recommendation']}")
    print("="*80 + "\n")
    
    return final_result

print("✅ Full pipeline ready")

## Cell 14: Results Visualization & Export

In [ ]:
def visualize_results(final_result: dict) -> None:
    """Display results in readable format."""
    print("\n" + "🎯 "*40)
    print("HASIL EVALUASI & RAGAS VALIDATION")
    print("🎯 "*40 + "\n")
    
    evaluation = final_result.get("evaluation", {})
    judge_validation = final_result.get("judge_validation", {})
    
    # Evaluation Scores
    print("📊 SKOR EVALUASI")
    print("─" * 60)
    scores = evaluation.get("scores", {})
    for key, label in SCORE_LABELS.items():
        score = scores.get(key, "N/A")
        weight = SCORE_WEIGHTS[key]
        print(f"  {label:40} | Skor: {score:3} (bobot: {weight}x)")
    
    final_score = evaluation.get("final_score", 0)
    print(f"\n  {'NILAI AKHIR':40} | {final_score} / 100")
    
    # RAGAS Confidence Scores
    print("\n\n📈 RAGAS CONFIDENCE SCORES")
    print("─" * 60)
    confidence = judge_validation.get("confidence_scores", {})
    for key, label in SCORE_LABELS.items():
        conf = confidence.get(key, 0)
        bar = "█" * int(conf * 10) + "░" * (10 - int(conf * 10))
        print(f"  {label:40} | {bar} {conf:.2f}")
    
    # RAGAS Metrics Details
    print("\n\n🔬 RAGAS METRICS DETAILS")
    print("─" * 60)
    ragas = judge_validation.get("ragas_metrics", {})
    for key, label in SCORE_LABELS.items():
        metrics = ragas.get(key, {})
        faith = metrics.get("faithfulness", 0)
        relevance = metrics.get("answer_relevance", 0)
        print(f"  {label}")
        print(f"    Faithfulness: {faith:.2f} | Answer Relevance: {relevance:.2f}")
        if metrics.get("faith_issues"):
            print(f"    Issues: {', '.join(metrics['faith_issues'])}")
        if metrics.get("relevance_gaps"):
            print(f"    Gaps: {', '.join(metrics['relevance_gaps'])}")
    
    # Flagged Criteria
    flagged = judge_validation.get("flagged_criteria", [])
    if flagged:
        print("\n\n⚠️ FLAGGED CRITERIA (Perlu Review)")
        print("─" * 60)
        for flag in flagged:
            print(f"  🚩 {SCORE_LABELS.get(flag['criterion'], flag['criterion'])}")
            print(f"     Reason: {flag['reason']}")
            print(f"     Faith Score: {flag['faith_score']} | Relevance: {flag['relevance_score']}")
            if flag.get("issues"):
                print(f"     Issues: {', '.join(flag['issues'][:2])}")
    
    # Final Recommendation
    print("\n\n⚖️ JUDGE VERDICT")
    print("─" * 60)
    print(f"  Quality Score: {judge_validation.get('overall_quality_score', 0)}")
    print(f"  Judge Notes: {judge_validation.get('judge_notes', '')}")
    print(f"  Recommendation: {final_result.get('final_recommendation', '')}")
    print(f"  Revision Needed: {'Yes ⚠️' if final_result.get('revision_needed') else 'No ✅'}")
    print(f"  Processing Time: {final_result.get('processing_time_seconds', 0):.2f}s")


def export_results_to_json(final_result: dict, output_path: str = None) -> str:
    """Export results to JSON file."""
    if output_path is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"evaluation_result_{timestamp}.json"
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_result, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Results exported to: {output_path}")
    return output_path


def export_results_to_csv(final_result: dict, output_path: str = None) -> str:
    """Export summary to CSV."""
    if output_path is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"evaluation_summary_{timestamp}.csv"
    
    evaluation = final_result.get("evaluation", {})
    judge_validation = final_result.get("judge_validation", {})
    
    rows = []
    scores = evaluation.get("scores", {})
    confidence = judge_validation.get("confidence_scores", {})
    ragas = judge_validation.get("ragas_metrics", {})
    
    for key, label in SCORE_LABELS.items():
        row = {
            "criterion": label,
            "score": scores.get(key, 0),
            "confidence": confidence.get(key, 0),
            "faithfulness": ragas.get(key, {}).get("faithfulness", 0),
            "answer_relevance": ragas.get(key, {}).get("answer_relevance", 0),
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False)
    print(f"✅ Summary exported to: {output_path}")
    return output_path


print("✅ Visualization & export functions ready")

## Cell 15: Sample Data & Testing

In [ ]:
# Sample makalah untuk testing
SAMPLE_MAKALAH = """
MAKALAH: STRATEGI PENGEMBANGAN SUMBER DAYA MANUSIA DI BPOM

Pendahuluan
Sumber daya manusia (SDM) adalah aset terpenting dalam organisasi. BPOM sebagai lembaga
pemerintah yang bertugas mengawasi obat dan makanan memerlukan SDM yang kompeten dan profesional.
Makalah ini menganalisis strategi pengembangan SDM di BPOM dalam mencapai visi dan misi organisasi.

Analisis dan Sintesis
Berdasarkan Rencana Strategis BPOM 2020-2024, terdapat lima sasaran strategis yang perlu didukung
oleh pengembangan SDM yang berkelanjutan. Pertama, meningkatkan kompetensi teknis melalui pelatihan
dan sertifikasi. Kedua, mengembangkan kepemimpinan untuk mempersiapkan generasi penerus.
Ketiga, memperkuat budaya kerja yang berorientasi pada layanan publik.

Analisis SWOT menunjukkan:
- Kekuatan: SDM tersebar di berbagai bidang keahlian
- Kelemahan: Keterbatasan anggaran pengembangan SDM
- Peluang: Dukungan dari kementerian terkait
- Ancaman: Persaingan untuk merekrut talenta terbaik

Rencana Strategis & Plan of Action
1. Program pelatihan berkelanjutan dengan fokus pada inovasi dan teknologi
2. Mentoring dari pemimpin senior untuk pengembangan karir
3. Kolaborasi dengan institusi pendidikan untuk peningkatan kompetensi
4. Evaluasi kinerja yang objektif dan transparan
5. Insentif untuk mendorong kinerja luar biasa

Kesimpulan
Pengembangan SDM yang terencana dan berkelanjutan adalah kunci kesuksesan BPOM dalam
mencapai visi dan misinya. Diperlukan komitmen dari seluruh jajaran manajemen untuk
merealisasikan strategi ini.
"""

SAMPLE_TEMA = """
KETENTUAN PENULISAN MAKALAH

1. Tema Umum: Strategi dan Inovasi dalam Pengembangan Organisasi

2. Sub-tema yang Direkomendasikan:
   - Strategi Pengembangan Sumber Daya Manusia
   - Inovasi Proses Operasional
   - Peningkatan Kualitas Layanan Publik
   - Kemitraan Strategis dan Kolaborasi

3. Struktur Makalah:
   a) Pendahuluan (2-3 halaman)
   b) Analisis dan Sintesis (4-5 halaman)
   c) Rencana Strategis (3-4 halaman)
   d) Rencana Aksi (2-3 halaman)
   e) Kesimpulan (1-2 halaman)

4. Kriteria Penilaian:
   - Kesesuaian dengan tema
   - Kedalaman analisis
   - Originalitas pemikiran
   - Sistematika penulisan
   - Penggunaan bahasa yang baik

5. Batasan: Maksimal 15 halaman, font Times New Roman 12pt, spasi 1.5
"""

print("✅ Sample data loaded")
print(f"   Makalah: {len(SAMPLE_MAKALAH)} karakter")
print(f"   Tema: {len(SAMPLE_TEMA)} karakter")

## Cell 16: Main Execution - Initialize RAG

In [ ]:
print("⏳ Initializing LightRAG (this may take a moment)...")
rag = run_async(initialize_rag())
print("✅ LightRAG initialized and ready for use")

## Cell 17: Run Full Pipeline

In [ ]:
# Run full pipeline with sample data
print("📚 Running full pipeline with sample data...")
final_result = run_async(full_pipeline(
    rag=rag,
    makalah_text=SAMPLE_MAKALAH,
    tema_text=SAMPLE_TEMA,
    selected_jabatan="Sekretaris Utama",
    query_mode="hybrid",
))

## Cell 18: Visualize & Export Results

In [ ]:
# Visualize results
visualize_results(final_result)

# Export results
json_path = export_results_to_json(final_result)
csv_path = export_results_to_csv(final_result)

print(f"\n📁 Results saved:")
print(f"   JSON: {json_path}")
print(f"   CSV: {csv_path}")

## Cell 19: Summary Statistics

In [ ]:
def compute_summary_stats(final_result: dict) -> dict:
    """Compute summary statistics from evaluation results."""
    evaluation = final_result.get("evaluation", {})
    judge_validation = final_result.get("judge_validation", {})
    
    scores = list(evaluation.get("scores", {}).values())
    confidence = list(judge_validation.get("confidence_scores", {}).values())
    
    summary = {
        "evaluation_metrics": {
            "final_score": evaluation.get("final_score", 0),
            "avg_criterion_score": np.mean(scores) if scores else 0,
            "min_score": np.min(scores) if scores else 0,
            "max_score": np.max(scores) if scores else 0,
            "std_dev": np.std(scores) if scores else 0,
        },
        "ragas_metrics": {
            "overall_quality_score": judge_validation.get("overall_quality_score", 0),
            "avg_confidence": np.mean(confidence) if confidence else 0,
            "min_confidence": np.min(confidence) if confidence else 0,
            "max_confidence": np.max(confidence) if confidence else 0,
        },
        "flagged_count": len(judge_validation.get("flagged_criteria", [])),
        "revision_needed": final_result.get("revision_needed", False),
        "processing_time_seconds": final_result.get("processing_time_seconds", 0),
    }
    
    return summary


# Print summary stats
stats = compute_summary_stats(final_result)
print("\n" + "="*60)
print("📊 SUMMARY STATISTICS")
print("="*60)
print("\n📈 Evaluation Metrics:")
for key, val in stats["evaluation_metrics"].items():
    print(f"  {key:20}: {val:.2f}")
print("\n🔬 RAGAS Metrics:")
for key, val in stats["ragas_metrics"].items():
    print(f"  {key:20}: {val:.2f}")
print(f"\n🚩 Flagged Count: {stats['flagged_count']}")
print(f"⚠️ Revision Needed: {stats['revision_needed']}")
print(f"⏱️ Processing Time: {stats['processing_time_seconds']:.2f}s")
print("="*60)